<a href="https://colab.research.google.com/github/shamseertk/ai-training/blob/main/ai_traning_01_Doc_Loading_splitting_embedding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [34]:
!pip install langchain_community faiss-cpu pypdf

In [17]:
from re import IGNORECASE
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceBgeEmbeddings, OpenAIEmbeddings
import faiss

import warnings
warnings.filterwarnings("ignore")

text_loader = TextLoader("./sample_data/nature_fiction_excerpts.txt")
text_document = text_loader.load()
print(text_document[:100])


[Document(metadata={'source': './sample_data/nature_fiction_excerpts.txt'}, page_content="The Whispering Canopy\n\nDeep within the Emerald Basin, where the ancient redwoods drank the morning mist before it could ever touch the valley floor, lived a cartographer named Elian who mapped the shifting paths of the wind. Unlike the scholars of the coastal cities who measured the earth in stone and boundary lines, Elian carried only spools of dyed silk, a brass anemometer, and a leather-bound ledger filled with sketches of migrating monarchs and cloud formations.\n\nEvery autumn, when the first frost silvered the fern fronds along the ridge, the forest seemed to hold its breath. The river slowed to a glassy crawl, reflecting the amber needles of the larches like a mirror of hammered copper. It was during this quiet transition that the Silver Stag appeared—a creature whose antlers were said to be woven from fallen starlight and winter birch.\n\nElian followed the stag's faint tracks through th

In [24]:
pdf_loader = PyPDFLoader("./sample_data/sample_pdf.pdf")
pdf_document = pdf_loader.load_and_split()
print(pdf_document[0])

page_content='•
Random Sample
T. P. Caravan
If you don't give me another piece of candy I'll cry. You'd be
surprised how loud I can cry. Mother wouldn't like that.
Thank you. I just love candy.
I'm very polite for my age; everybody says so. I can get
more candy that way. Old ladies are best. I'm also a·very in-
telligent little girl, but I suppose you found that out from
your tests. They gave me the same kind of tests, but they
didn't give me any candy, so I was bad and didn't answer
anything right.
Thank you. I'll take two this time. Do you have any hard
candies? The heat's melted these chocolates a little.
My father says to get all I can out of you, because all you
Viennese head-thumpers are quacks. He says you cost an aw-
ful lot of money. He says only an old fraud would have a
beard like a billy goat. He says ...
Are you getting angry?
All right, then, if you give me just one more pIece of
candy I'll tell you all about it.
Merci. That's French, you know.
My brother Johnny and I wer

In [27]:
doc_splitter = RecursiveCharacterTextSplitter(chunk_size = 1024, chunk_overlap = 64)
pdf_splitted_chunks = doc_splitter.split_documents(pdf_document)
print(len(pdf_splitted_chunks))

10


In [29]:
MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"
hf_embed = HuggingFaceBgeEmbeddings(model_name=MODEL_NAME)
text = pdf_splitted_chunks[0].page_content
hf_embed_result = hf_embed.embed_documents([text])
print(len(hf_embed_result[0]))


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

768


In [35]:
openai_embedding = OpenAIEmbeddings()
openai_embed_result = openai_embedding.embed_documents([text])
print(len(openai_embed_result[0]))

ValidationError: 1 validation error for OpenAIEmbeddings
  Value error, Did not find openai_api_key, please add an environment variable `OPENAI_API_KEY` which contains it, or pass `openai_api_key` as a named parameter. [type=value_error, input_value={'model_kwargs': {}, 'cli...20, 'http_client': None}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/value_error

In [39]:
faiss_instance = FAISS.from_documents(pdf_splitted_chunks, hf_embed)


In [40]:
search_result = faiss_instance.similarity_search_with_score("What did Johny in the war?", k=2)
print(search_result)

[(Document(id='82d37206-8f61-4ee5-a776-be6fd4d1462e', metadata={'producer': 'Image Recognition Integrated Systems, Autoformat5,0,0,229', 'creator': 'I.R.I.S.', 'creationdate': '2012-10-26T13:47:41-04:00', 'source': './sample_data/sample_pdf.pdf', 'total_pages': 4, 'page': 3, 'page_label': '4'}, page_content="to stomp ants. It was very funny. One of them got sick. John-\nny and I were still jumping up and down, stomping ants,\nwhen they took off. I hated them. They were nasty; they\ndidn't bow down to me.\nThat's all. Nothing else happened.\nFather says not to take up too much of your expensive old\ntime. He says no honest man could afford a penthouse for\nhis office. You have a very nice view, don't you? You can see\nall over the city from here.\nMy, isn't it hot? I wish I had a refrigerator to keep my\ncandy in.\nLook there. Look at the fires springing up across the river.\nAren't they pretty? Look. Look. And some on this side.\n. Take me away from here. It's too hot.\nLook at the sun